# nn.Module — The Base Class for All Models

`nn.Module` is the base class for every neural network in PyTorch.
All built-in layers (nn.Linear, nn.Conv2d, etc.) extend it.
Your custom models extend it too.

**Java analogy:** like implementing an interface or extending an abstract class.
The framework calls your `forward()` method — like implementing a `service()` method.

In [ ]:
import torch
import torch.nn as nn

# Simplest possible model — one linear layer
# nn.Linear(in_features, out_features) = W matrix + b bias
# internally does: y = xW^T + b

linear = nn.Linear(in_features=3, out_features=2)

print('weight shape:', linear.weight.shape)   # [out, in] = [2, 3]
print('bias shape:  ', linear.bias.shape)     # [out] = [2]
print()
print('weight:\n', linear.weight)
print('bias:  ', linear.bias)

## Defining your own model

You always:
1. Inherit from `nn.Module`
2. Define layers in `__init__`
3. Define the forward pass in `forward()`

PyTorch calls `forward()` when you do `model(input)` — same as `__call__` in Python.

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()   # MUST call parent __init__

        # Define layers as attributes — nn.Module tracks these automatically
        self.layer1 = nn.Linear(input_size, hidden_size)
        self.relu   = nn.ReLU()
        self.layer2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        # Define how data flows through the model
        x = self.layer1(x)   # linear transformation
        x = self.relu(x)     # non-linearity
        x = self.layer2(x)   # output layer
        return x

# Create model
model = SimpleNet(input_size=4, hidden_size=8, output_size=2)
print(model)

In [ ]:
# Run a forward pass
x = torch.randn(1, 4)   # batch of 1, 4 features
out = model(x)          # calls model.forward(x) internally

print('input shape: ', x.shape)
print('output shape:', out.shape)   # [1, 2]

## parameters() — accessing all weights

`model.parameters()` returns all trainable weights. This is what you pass to the optimizer.

In [ ]:
# Count total trainable parameters
total = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {total}')

# See each parameter's shape
for name, param in model.named_parameters():
    print(f'{name:20s} shape: {param.shape}  | numel: {param.numel()}')

## nn.Sequential — shortcut for linear stacks

When layers just flow one after another, `nn.Sequential` avoids writing a full class.

In [ ]:
# Equivalent to SimpleNet above but shorter
model_seq = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

print(model_seq)

x = torch.randn(5, 4)   # batch of 5
out = model_seq(x)
print('output shape:', out.shape)   # [5, 2]

## train() vs eval() — critical for inference

Some layers (Dropout, BatchNorm) behave differently during training vs inference.
Always call `model.eval()` before running predictions.

In [ ]:
class NetWithDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer   = nn.Linear(4, 8)
        self.dropout = nn.Dropout(p=0.5)   # randomly zeros 50% of neurons during training
        self.out     = nn.Linear(8, 2)

    def forward(self, x):
        x = self.layer(x)
        x = self.dropout(x)   # active in train mode, identity in eval mode
        return self.out(x)

model = NetWithDropout()
x = torch.randn(1, 4)

# Training mode — dropout is active
model.train()
out_train = model(x)
print('train mode output:', out_train)

# Eval mode — dropout disabled, deterministic output
model.eval()
with torch.no_grad():
    out_eval = model(x)
print('eval mode output: ', out_eval)
print('outputs differ:   ', not torch.allclose(out_train, out_eval))

## Saving and loading models

In [ ]:
# Save — only save the weights (state_dict), not the model class
torch.save(model.state_dict(), 'model_weights.pt')

# Load — create model first, then load weights into it
loaded_model = NetWithDropout()
loaded_model.load_state_dict(torch.load('model_weights.pt', weights_only=True))
loaded_model.eval()

print('Model loaded successfully')
print(loaded_model.state_dict().keys())